# PKG customer metrics v1: one row per customer per month

**PNC Treasury Management · Data Science · Payment Knowledge Graph**

**Inputs**
- the edge tables built by `pkg_edges_2024_backfill.ipynb`: `pkg_edge_monthly`, `pkg_edge_daily`, `pkg_cpty_dim`, `pkg_cpty_entity_map`;
- the MDM customer table `dsihd01p_dsi.neo4j_customer`.

**Output** (database `bdahd01p_dlcdi1_cdi_tm`)

| Table | Grain | Purpose |
|---|---|---|
| `pkg_cust_dim` | one row per `mdm_id` (latest `load_dt`) | customer identity: power ids, name, party type, NAICS, TM flag, family size |
| `pkg_cust_pair_month` | customer × direction × scope × counterparty, per month | intermediate; drives breadth, concentration and churn, reusable for cohort windows |
| `pkg_cust_metrics_monthly` | **one row per customer per month** | identity columns first, then every metric |
| `pkg_cust_metrics_dictionary` | one row per column of the metrics table | definition, family, scope, direction |

## Conventions
- **Column names:** `{metric}_{scope}_{dir}`.
  - **Scope:** `int` = the counterparty is another PNC customer; `ext` = an external counterparty; `self` = the same `mdm_id` on both sides; `all` = `int` + `ext` (self-transfers excluded).
  - **Direction:** `out` = the customer pays; `in` = the customer receives.
- **Counterparty identity:** external counterparties are counted at entity level `ent_l1`, meaning raw id with the 6C P2P ids folded into their 5C id. On the full history, name merging (`ent_l2`) moved per-customer breadth or churn by 10% or more for fewer than 2.5% of customers, so `ent_l1` is used. Internal counterparties are counted by `mdm_id`.
- **Customer attributes are current-state:** the latest `load_dt` per `mdm_id` is applied to every month.
- **TM customer** = has a `customer_pwr_id`. `master_pwr_id` is the ultimate parent. Metrics are computed for **every** customer in the edges; filter on `is_tm` for the TM population.
- **NAICS applies to TM customers.** A null NAICS outside TM is expected. `naics_status` = valid / placeholder (`******`, UNKNOWN, …) / missing.
- **Customers with no activity in a month but with counterparties the month before** get a row with zero flows. That makes going silent visible through `lost_*`.
- **Empty values:** shares are NULL when their denominator is 0, and amounts and counts are 0 when there's no activity. Sector-mix metrics are NULL below 5 internal counterparties with a valid NAICS.

**How to run.** Run all cells. Months already built are skipped unless `FORCE`. The default is a two-month trial; for the full history set
`RUN_MONTHS = month_range("2024-01", LAST_COMPLETE)`. Churn for the first requested month needs the previous month's pairs, which are built automatically when that month's edges exist.

In [ ]:
# ---- Config -------------------------------------------------------------------------------
import calendar, datetime as dt
def month_range(a, b):
    y, m = map(int, a.split("-")); y2, m2 = map(int, b.split("-")); out = []
    while (y, m) <= (y2, m2):
        out.append(f"{y:04d}-{m:02d}"); y, m = (y + 1, 1) if m == 12 else (y, m + 1)
    return out
_t = dt.date.today().replace(day=1) - dt.timedelta(days=1)
LAST_COMPLETE = f"{_t.year:04d}-{_t.month:02d}"

CUST_TABLE = "dsihd01p_dsi.neo4j_customer"
TGT_DB     = "bdahd01p_dlcdi1_cdi_tm"
PREFIX     = "pkg_"

RUN_MONTHS = month_range("2026-07", "2026-08")      # trial.  Full: month_range("2024-01", LAST_COMPLETE)
FORCE          = False                               # True = rebuild months already written
BUILD_CUST_DIM = True                                # rebuild pkg_cust_dim from the latest load_dt

RAILS = ["ACH", "WIRE", "RTP_PRT", "RTP_P2P", "CHECK", "PCARD"]
RAIL_OF_DIGIT = {"1": "ACH", "2": "ACH", "3": "ACH", "4": "WIRE", "5": "RTP_PRT", "6": "RTP_P2P",
                 "7": "CHECK", "8": "DEBIT_CARD", "9": "PCARD"}
NAICS_PLACEHOLDERS = ["******", "-1", "0", "00", "000000", "999999", "UNKNOWN", "N/A", "NA", "NULL", "NONE"]
MIN_NAICS_CP = 5                                     # sector-mix metrics need ≥ this many internal counterparties with valid NAICS

FILES = {"cust_pair_month": 8, "cust_metrics_monthly": 4}
CODEC = "snappy"
SHUFFLE_PARTITIONS = 400

In [ ]:
# ---- Setup. NumPy shim: Cloudera SPARK3 PySpark predates NumPy 1.24/2.0 aliases used by toPandas()
import numpy as np, warnings
for _alias, _target in {"object0": np.object_, "bool8": np.bool_, "int0": np.intp, "uint0": np.uintp}.items():
    if not hasattr(np, _alias):
        setattr(np, _alias, _target)
if "bool" not in np.__dict__:
    np.bool = np.bool_
warnings.filterwarnings("ignore", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"pyspark.*")

from pyspark.sql import SparkSession, Window, functions as F
from pyspark import StorageLevel
from IPython.display import display
from contextlib import contextmanager
import pandas as pd, time

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
print("Spark", spark.version, "| months:", RUN_MONTHS[0], "→", RUN_MONTHS[-1], f"({len(RUN_MONTHS)})")
pd.set_option("display.max_rows", 300); pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 250);    pd.set_option("display.max_colwidth", 120)
MD = StorageLevel.MEMORY_AND_DISK

def show(pdf, title, pct=(), money=()):
    print(f"\n■ {title}   [{len(pdf):,} rows]")
    fmt = {}
    for c in pdf.columns:
        if c in pct:                                fmt[c] = "{:.2%}"
        elif c in money:                            fmt[c] = "${:,.0f}"
        elif pd.api.types.is_integer_dtype(pdf[c]): fmt[c] = "{:,}"
        elif pd.api.types.is_float_dtype(pdf[c]):   fmt[c] = "{:,.3f}"
    sty = pdf.style.format(fmt, na_rep="—")
    try:    sty = sty.hide(axis="index")
    except AttributeError: sty = sty.hide_index()
    display(sty)

def nz(c):
    s = F.trim(F.col(c).cast("string"))
    return F.when(s == "", F.lit(None)).otherwise(s)

TIMINGS = []
@contextmanager
def timed(stage, month=None):
    t0 = time.time(); yield
    s = time.time() - t0
    TIMINGS.append({"month": month, "stage": stage, "seconds": s})
    print(f"  ⏱ {month or '':<8} {stage}: {s/60:,.1f} min")

HCONF = spark._jsc.hadoopConfiguration()
JPath = spark._jvm.org.apache.hadoop.fs.Path
fs_for = lambda p: JPath(p).getFileSystem(HCONF)
exists = lambda p: fs_for(p).exists(JPath(p))
def prev_month(m):
    y, mo = map(int, m.split("-"))
    return f"{y - 1:04d}-12" if mo == 1 else f"{y:04d}-{mo - 1:02d}"
GB = 1e9

DB_LOC = [r[1] for r in spark.sql(f"DESCRIBE DATABASE {TGT_DB}").collect() if r[0].lower() == "location"][0].rstrip("/")
KEYS_T = ["edge_monthly", "edge_daily", "cpty_dim", "cpty_entity_map",
          "cust_dim", "cust_pair_month", "cust_metrics_monthly", "cust_metrics_dictionary"]
T = {k: f"{TGT_DB}.{PREFIX}{k}" for k in KEYS_T}
P = {k: f"{DB_LOC}/{PREFIX}{k}" for k in KEYS_T}
for k in ("edge_monthly", "edge_daily", "cpty_dim", "cpty_entity_map"):
    assert exists(P[k]), f"{T[k]} not found: run pkg_edges_2024_backfill.ipynb first"

def ensure_table(key, schema, partitioned):
    cols = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in schema.fields if f.name != "month")
    part = " PARTITIONED BY (month STRING)" if partitioned else ""
    spark.sql(f"CREATE EXTERNAL TABLE IF NOT EXISTS {T[key]} ({cols}){part} STORED AS PARQUET LOCATION '{P[key]}'")

def add_partition(key, m):
    spark.sql(f"ALTER TABLE {T[key]} ADD IF NOT EXISTS PARTITION (month='{m}') LOCATION '{P[key]}/month={m}'")

def month_done(key, m):
    return exists(f"{P[key]}/month={m}/_SUCCESS")

def write_month(df, key, m, part_col, sort_cols):
    (df.repartition(FILES[key], part_col).sortWithinPartitions(*sort_cols)
       .write.mode("overwrite").option("compression", CODEC).parquet(f"{P[key]}/month={m}"))
    ensure_table(key, df.schema, True); add_partition(key, m)

def write_full(df, key):
    """Non-partitioned table: rewrite files, then drop + re-create so the schema always matches (metadata only)."""
    df.write.mode("overwrite").option("compression", CODEC).parquet(P[key])
    spark.sql(f"DROP TABLE IF EXISTS {T[key]}")
    ensure_table(key, df.schema, False); spark.catalog.refreshTable(T[key])

def size_of(path):
    if not exists(path): return 0, 0
    c = fs_for(path).getContentSummary(JPath(path))
    return c.getLength(), c.getFileCount()
print("target:", DB_LOC)

## §1 Customer dimension: `pkg_cust_dim`

This takes one row per `mdm_id` (the latest `load_dt`; on a tie, the row with a `customer_pwr_id` wins) and adds NAICS status, the 2-digit sector, the TM flag and family size (the number of customers under the same `master_pwr_id`).

The checks are:
- that `mdm_id` ↔ `customer_pwr_id` is one-to-one;
- how often identity changed across `load_dt` versions;
- NAICS status by TM flag and party type;
- family structure;
- how many edge customers the dimension covers.

In [ ]:
CUST_COLS = ["mdm_id", "customer_pwr_id", "master_pwr_id", "customer_name", "party_type",
             "customer_start_dt", "naics_cd", "naics_desc", "load_dt"]
_have = {c.lower() for c in spark.table(CUST_TABLE).columns}
assert not (set(CUST_COLS) - _have), f"{CUST_TABLE} missing {set(CUST_COLS) - _have}"

if BUILD_CUST_DIM or not exists(P["cust_dim"]):
    with timed("customer dimension"):
        raw = (spark.table(CUST_TABLE).select(*[nz(c).alias(c) for c in CUST_COLS])
                    .filter(F.col("mdm_id").isNotNull())
                    .withColumn("party_type", F.upper("party_type")))
        w = Window.partitionBy("mdm_id").orderBy(F.col("load_dt").desc_nulls_last(),
                                                 F.col("customer_pwr_id").isNotNull().desc())
        latest = (raw.withColumn("n_load_versions", F.count("*").over(Window.partitionBy("mdm_id")))
                     .withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn"))
        nc = F.upper(F.col("naics_cd"))
        latest = (latest
            .withColumn("naics_status", F.when(nc.isNull(), "missing")
                                         .when(nc.isin(NAICS_PLACEHOLDERS) | ~nc.rlike(r"^[0-9]{2}"), "placeholder")
                                         .otherwise("valid"))
            .withColumn("naics2", F.when(F.col("naics_status") == "valid", F.substring("naics_cd", 1, 2)))
            .withColumn("is_tm", F.col("customer_pwr_id").isNotNull()))
        fam = latest.filter(F.col("master_pwr_id").isNotNull()).groupBy("master_pwr_id").agg(F.count("*").alias("family_size"))
        dim = (latest.join(fam, "master_pwr_id", "left")
                     .select("mdm_id", "customer_pwr_id", "master_pwr_id", "customer_name", "party_type",
                             "naics_cd", "naics_desc", "naics2", "naics_status", "is_tm", "family_size",
                             "customer_start_dt", "load_dt", "n_load_versions"))
        write_full(dim, "cust_dim")

CD = spark.table(T["cust_dim"]).persist(MD)
q = CD.agg(F.count("*").alias("mdm_ids"), F.sum(F.col("is_tm").cast("int")).alias("tm_customers"),
           F.countDistinct("master_pwr_id").alias("master_pwr_ids"),
           F.avg((F.col("n_load_versions") > 1).cast("double")).alias("multi_version_share"),
           F.max("load_dt").alias("latest_load_dt")).toPandas()
show(q, f"{T['cust_dim']}", pct=("multi_version_share",))

viol = (CD.filter(F.col("customer_pwr_id").isNotNull()).groupBy("customer_pwr_id")
          .agg(F.count("*").alias("n_mdm")).filter("n_mdm > 1"))
hist = (spark.table(CUST_TABLE).select(nz("mdm_id").alias("mdm_id"), nz("customer_pwr_id").alias("pwr"))
             .filter(F.col("mdm_id").isNotNull() & F.col("pwr").isNotNull())
             .groupBy("mdm_id").agg(F.countDistinct("pwr").alias("k")).filter("k > 1"))
show(pd.DataFrame([{"pwr_ids_shared_by_2plus_mdm (latest)": viol.count(),
                    "mdm_ids_with_2plus_pwr_over_history": hist.count()}]),
     "One-to-one check mdm_id ↔ customer_pwr_id (expected 0 / small)")

show(CD.groupBy("is_tm", "party_type", "naics_status").count().orderBy("is_tm", "party_type", "naics_status").toPandas(),
     "NAICS status by TM flag and party type")
show(CD.filter(F.col("is_tm")).agg(
        F.avg((F.col("customer_pwr_id") == F.col("master_pwr_id")).cast("double")).alias("tm_is_own_master"),
        F.expr("percentile_approx(family_size, array(0.5, 0.9, 0.99))").alias("family_size_p50_p90_p99"),
        F.max("family_size").alias("family_size_max")).toPandas(),
     "Families (TM customers)", pct=("tm_is_own_master",))

## §2 Monthly build

For each month:
1. **Perspective rows.** Every monthly edge is read from each customer side it touches: `out` for the payer, `in` for the payee. An internal edge therefore yields two rows.
2. **Counterparty attributes.**
   - External counterparties map to `ent_l1` and take that entity's FI and business-name flag from `pkg_cpty_dim`.
   - Internal counterparties take TM flag, party type and NAICS2 from `pkg_cust_dim`.
3. **Pairs:** customer × direction × scope × counterparty, written to `pkg_cust_pair_month`.
4. **Metrics** from the perspective rows (flows, rails, compositions), the pairs (breadth, concentration, FI, sectors, churn against the previous month's pairs) and the daily edges (active days).
5. **Exact reconciliation:** the sum of each customer's outflows (int + ext + self) must equal all edge dollars paid by customers, and the same for inflows.

In [ ]:
RAIL_MAP = F.create_map(*[F.lit(x) for kv in RAIL_OF_DIGIT.items() for x in kv])
S3, S2, D = ["int", "ext", "self"], ["int", "ext"], ["in", "out"]
DIM_SLIM = spark.table(T["cpty_dim"]).select(F.col("cpty_id").alias("d_id"), F.col("fi").alias("o_fi"),
                                             F.col("is_business").alias("o_biz"))
XW = (spark.table(T["cpty_entity_map"]).filter(F.col("ent_l1") != F.col("cpty_id"))
           .select(F.col("cpty_id").alias("xw_from"), F.col("ent_l1").alias("xw_to")))
CATTR = CD.select(F.col("mdm_id").alias("c_id"), F.col("is_tm").alias("o_is_tm"),
                  F.col("party_type").alias("o_party_type"), F.col("naics2").alias("o_naics2"))
OWN = CD.select(F.col("mdm_id").alias("cust"), F.col("naics2").alias("own_naics2"))

def perspective(m):
    mon = spark.read.parquet(f"{P['edge_monthly']}/month={m}")
    keep = ["category", "is_self", "amount", "n_txn"]
    out_ = mon.filter(F.col("src_kind") == "CUST").select(F.col("src_id").alias("cust"), F.lit("out").alias("dir"),
              F.col("dst_id").alias("oid"), F.col("dst_kind").alias("okind"), *keep)
    in_  = mon.filter(F.col("dst_kind") == "CUST").select(F.col("dst_id").alias("cust"), F.lit("in").alias("dir"),
              F.col("src_id").alias("oid"), F.col("src_kind").alias("okind"), *keep)
    p = (out_.unionByName(in_)
             .withColumn("scope", F.when(F.col("is_self"), "self").when(F.col("okind") == "CUST", "int").otherwise("ext"))
             .withColumn("rail", F.element_at(RAIL_MAP, F.substring("category", 1, 1))))
    ext = (p.filter(F.col("scope") == "ext")
            .join(XW, F.col("oid") == F.col("xw_from"), "left")
            .withColumn("other", F.coalesce("xw_to", "oid")).drop("xw_from", "xw_to")
            .join(DIM_SLIM, F.col("other") == F.col("d_id"), "left").drop("d_id"))
    intr = (p.filter(F.col("scope") != "ext").withColumn("other", F.col("oid"))
             .join(CATTR, F.col("other") == F.col("c_id"), "left").drop("c_id"))
    return ext.unionByName(intr, allowMissingColumns=True).drop("oid", "okind", "is_self")

def build_pairs(m, persp):
    pr = (persp.filter(F.col("other").isNotNull())
               .groupBy("cust", "dir", "scope", "other")
               .agg(F.sum("amount").cast("decimal(22,2)").alias("amount"), F.sum("n_txn").cast("long").alias("n_txn"),
                    F.first("o_fi", ignorenulls=True).alias("o_fi"),
                    F.max(F.col("o_biz").cast("int")).cast("boolean").alias("o_biz"),
                    F.max(F.col("o_is_tm").cast("int")).cast("boolean").alias("o_is_tm"),
                    F.first("o_party_type", ignorenulls=True).alias("o_party_type"),
                    F.first("o_naics2", ignorenulls=True).alias("o_naics2")))
    write_month(pr, "cust_pair_month", m, "cust", ["cust", "dir", "other"])

In [ ]:
def metrics_for(m, persp, PR):
    amt = F.col("amount").cast("double")
    c = lambda s, d: (F.col("scope") == s) & (F.col("dir") == d)

    # ---- flows, compositions, rails (perspective rows) ---------------------------------------
    ag = []
    for s in S3:
        for d in D:
            ag += [F.sum(F.when(c(s, d), F.col("amount"))).alias(f"amt_{s}_{d}"),
                   F.sum(F.when(c(s, d), F.col("n_txn"))).alias(f"ntxn_{s}_{d}")]
    for d in D:
        ag += [F.sum(F.when(c("ext", d) & F.col("o_biz"), amt)).alias(f"_biz_{d}"),
               F.sum(F.when(c("int", d) & F.col("o_is_tm"), amt)).alias(f"_tm_{d}"),
               F.sum(F.when(c("int", d) & (F.col("o_party_type") == "O"), amt)).alias(f"_org_{d}")]
        ag += [F.sum(F.when((F.col("dir") == d) & (F.col("scope") != "self") & (F.col("rail") == r), amt)).alias(f"_rail_{r}_{d}")
               for r in RAILS]
    flow = persp.groupBy("cust").agg(*ag)

    # ---- breadth and concentration (pairs, self excluded) ------------------------------------
    NS = PR.filter(F.col("scope") != "self").withColumn("a", amt)
    ag = [F.sum(F.when(c(s, d), 1).otherwise(0)).alias(f"ncpty_{s}_{d}") for s in S2 for d in D]
    for d in D:
        ag += [F.sum(F.when(F.col("dir") == d, F.col("a"))).alias(f"_tot_{d}"),
               F.max(F.when(F.col("dir") == d, F.col("a"))).alias(f"_max_{d}"),
               F.sum(F.when(F.col("dir") == d, F.col("a") * F.col("a"))).alias(f"_sq_{d}")]
    conc = NS.groupBy("cust").agg(*ag)

    # ---- external banks ------------------------------------------------------------------------
    fi = (NS.filter((F.col("scope") == "ext") & F.col("o_fi").isNotNull())
            .groupBy("cust", "dir", "o_fi").agg(F.sum("a").alias("a"))
            .groupBy("cust").agg(*[x for d in D for x in (
                F.sum(F.when(F.col("dir") == d, 1).otherwise(0)).alias(f"n_fi_{d}"),
                F.max(F.when(F.col("dir") == d, F.col("a"))).alias(f"_fimax_{d}"),
                F.sum(F.when(F.col("dir") == d, F.col("a"))).alias(f"_fitot_{d}"))]))

    # ---- sector mix of internal counterparties ----------------------------------------------------
    nx = NS.filter((F.col("scope") == "int") & F.col("o_naics2").isNotNull()).join(OWN, "cust", "left")
    g = nx.groupBy("cust", "dir", "o_naics2").agg(F.sum("a").alias("a"), F.count("*").alias("n"))
    g = g.withColumn("p", F.col("a") / F.sum("a").over(Window.partitionBy("cust", "dir")))
    sect = g.groupBy("cust").agg(*[x for d in D for x in (
                F.sum(F.when(F.col("dir") == d, 1).otherwise(0)).alias(f"_nn_{d}"),
                F.sum(F.when(F.col("dir") == d, F.col("n")).otherwise(0)).alias(f"_ncp_{d}"),
                F.max(F.when(F.col("dir") == d, F.col("p"))).alias(f"_ntop_{d}"),
                F.sum(F.when(F.col("dir") == d, -F.col("p") * F.log(F.col("p")))).alias(f"_nent_{d}"))])
    same = nx.filter(F.col("own_naics2").isNotNull()).groupBy("cust").agg(*[x for d in D for x in (
                F.sum(F.when((F.col("dir") == d) & (F.col("o_naics2") == F.col("own_naics2")), F.col("a")).otherwise(0.0)).alias(f"_same_{d}"),
                F.sum(F.when(F.col("dir") == d, F.col("a"))).alias(f"_valid_{d}"))])

    # ---- churn vs previous month (entity level, self excluded) -------------------------------
    pm, K = prev_month(m), ["cust", "dir", "other"]
    has_prev = month_done("cust_pair_month", pm)
    base = persp.select("cust").distinct()
    if has_prev:
        prv = (spark.read.parquet(f"{P['cust_pair_month']}/month={pm}").filter(F.col("scope") != "self")
                    .select(*K).distinct())
        base = base.unionByName(prv.select("cust")).distinct()
        cur = NS.select(*K, "a")
        new = cur.join(prv, K, "left_anti").groupBy("cust").agg(*[x for d in D for x in (
                  F.sum(F.when(F.col("dir") == d, 1).otherwise(0)).alias(f"new_cpty_{d}"),
                  F.sum(F.when(F.col("dir") == d, F.col("a"))).alias(f"_newa_{d}"))])
        ret = cur.join(prv, K, "left_semi").groupBy("cust").agg(
                  *[F.sum(F.when(F.col("dir") == d, 1).otherwise(0)).alias(f"retained_cpty_{d}") for d in D])
        lost = prv.join(cur.select(*K), K, "left_anti").groupBy("cust").agg(
                  *[F.sum(F.when(F.col("dir") == d, 1).otherwise(0)).alias(f"lost_cpty_{d}") for d in D])

    # ---- active days (daily edges) --------------------------------------------------------------
    dd = spark.read.parquet(f"{P['edge_daily']}/month={m}").filter(~F.col("is_self"))
    act = (dd.filter(F.col("src_kind") == "CUST").select(F.col("src_id").alias("cust"), F.lit("out").alias("dir"), "txn_date")
             .unionByName(dd.filter(F.col("dst_kind") == "CUST").select(F.col("dst_id").alias("cust"), F.lit("in").alias("dir"), "txn_date"))
             .distinct().groupBy("cust").agg(
                 *[F.sum(F.when(F.col("dir") == d, 1).otherwise(0)).alias(f"active_days_{d}") for d in D]))

    X = base.join(flow, "cust", "left").join(conc, "cust", "left").join(fi, "cust", "left") \
            .join(sect, "cust", "left").join(same, "cust", "left").join(act, "cust", "left")
    if has_prev:
        X = X.join(new, "cust", "left").join(ret, "cust", "left").join(lost, "cust", "left")
    else:
        for d in D:
            X = (X.withColumn(f"new_cpty_{d}", F.lit(None).cast("long")).withColumn(f"_newa_{d}", F.lit(None).cast("double"))
                  .withColumn(f"retained_cpty_{d}", F.lit(None).cast("long")).withColumn(f"lost_cpty_{d}", F.lit(None).cast("long")))

    # ---- derived columns -------------------------------------------------------------------------
    z  = lambda col: F.coalesce(F.col(col).cast("double"), F.lit(0.0))
    sd = lambda num, den: F.when(den > 0, num / den)
    for s in S3:
        for d in D:
            X = (X.withColumn(f"amt_{s}_{d}", F.coalesce(F.col(f"amt_{s}_{d}"), F.lit(0)).cast("decimal(22,2)"))
                  .withColumn(f"ntxn_{s}_{d}", F.coalesce(F.col(f"ntxn_{s}_{d}"), F.lit(0)).cast("long")))
    for s in S2:
        for d in D:
            X = X.withColumn(f"ncpty_{s}_{d}", F.coalesce(F.col(f"ncpty_{s}_{d}"), F.lit(0)).cast("long"))
    for d in D:
        X = (X.withColumn(f"amt_all_{d}", (F.col(f"amt_int_{d}") + F.col(f"amt_ext_{d}")).cast("decimal(22,2)"))
              .withColumn(f"ntxn_all_{d}", F.col(f"ntxn_int_{d}") + F.col(f"ntxn_ext_{d}"))
              .withColumn(f"ncpty_all_{d}", F.col(f"ncpty_int_{d}") + F.col(f"ncpty_ext_{d}"))
              .withColumn(f"active_days_{d}", F.coalesce(F.col(f"active_days_{d}"), F.lit(0)).cast("int")))
        A, E, I = z(f"amt_all_{d}"), z(f"amt_ext_{d}"), z(f"amt_int_{d}")
        X = (X.withColumn(f"ext_share_{d}", sd(E, A))
              .withColumn(f"top1_share_{d}", sd(z(f"_max_{d}"), z(f"_tot_{d}")))
              .withColumn(f"hhi_{d}", sd(z(f"_sq_{d}"), z(f"_tot_{d}") * z(f"_tot_{d}")))
              .withColumn(f"n_fi_{d}", F.coalesce(F.col(f"n_fi_{d}"), F.lit(0)).cast("long"))
              .withColumn(f"top_fi_share_{d}", sd(z(f"_fimax_{d}"), z(f"_fitot_{d}")))
              .withColumn(f"fi_coverage_{d}", sd(z(f"_fitot_{d}"), E))
              .withColumn(f"ext_business_share_{d}", sd(z(f"_biz_{d}"), E))
              .withColumn(f"int_tm_share_{d}", sd(z(f"_tm_{d}"), I))
              .withColumn(f"int_org_share_{d}", sd(z(f"_org_{d}"), I)))
        guard = z(f"_ncp_{d}") >= MIN_NAICS_CP
        X = (X.withColumn(f"n_naics2_{d}", F.when(guard, F.col(f"_nn_{d}")).cast("long"))
              .withColumn(f"naics2_top_share_{d}", F.when(guard, F.col(f"_ntop_{d}")))
              .withColumn(f"naics2_entropy_{d}", F.when(guard, F.col(f"_nent_{d}")))
              .withColumn(f"same_naics2_share_{d}", sd(z(f"_same_{d}"), z(f"_valid_{d}"))))
        for r in RAILS:
            X = X.withColumn(f"share_{r}_{d}", sd(z(f"_rail_{r}_{d}"), A))
        X = X.withColumn(f"n_rails_{d}", sum((z(f"_rail_{r}_{d}") > 0).cast("int") for r in RAILS))
        X = (X.withColumn(f"new_cpty_{d}", F.col(f"new_cpty_{d}").cast("long"))
              .withColumn(f"retained_cpty_{d}", F.col(f"retained_cpty_{d}").cast("long"))
              .withColumn(f"lost_cpty_{d}", F.col(f"lost_cpty_{d}").cast("long"))
              .withColumn(f"new_amt_share_{d}", sd(z(f"_newa_{d}"), A)))
        if has_prev:
            X = (X.withColumn(f"new_cpty_{d}", F.coalesce(F.col(f"new_cpty_{d}"), F.lit(0)))
                  .withColumn(f"retained_cpty_{d}", F.coalesce(F.col(f"retained_cpty_{d}"), F.lit(0)))
                  .withColumn(f"lost_cpty_{d}", F.coalesce(F.col(f"lost_cpty_{d}"), F.lit(0))))
    X = (X.withColumn("net_flow_all", (F.col("amt_all_in") - F.col("amt_all_out")).cast("decimal(22,2)"))
          .withColumn("net_flow_ext", (F.col("amt_ext_in") - F.col("amt_ext_out")).cast("decimal(22,2)"))
          .withColumn("passthrough_ratio", sd(F.least(z("amt_all_in"), z("amt_all_out")), F.greatest(z("amt_all_in"), z("amt_all_out")))))

    ident = CD.select(F.col("mdm_id").alias("cust"), "customer_pwr_id", "master_pwr_id", "customer_name", "party_type",
                      "naics_cd", "naics_desc", "naics2", "naics_status", "is_tm", "family_size", "customer_start_dt")
    X = X.join(ident, "cust", "left").withColumn("in_cust_dim", F.col("is_tm").isNotNull()) \
         .withColumn("is_tm", F.coalesce(F.col("is_tm"), F.lit(False))).withColumnRenamed("cust", "mdm_id")
    return X.select(*IDENTITY_COLS, *[c for c, *_ in METRIC_SPEC])

In [ ]:
# ---- Column order and data dictionary (single source of truth) ---------------------------------
IDENTITY_COLS = ["mdm_id", "customer_pwr_id", "master_pwr_id", "customer_name", "party_type", "naics_cd", "naics_desc",
                 "naics2", "naics_status", "is_tm", "family_size", "customer_start_dt", "in_cust_dim"]
IDENTITY_DEF = {
    "mdm_id": "MDM party id (bank-wide customer key)",
    "customer_pwr_id": "TM customer power id; null = not a TM customer",
    "master_pwr_id": "ultimate parent power id of the customer's family",
    "customer_name": "customer name (latest load_dt)",
    "party_type": "P = person, O = organisation",
    "naics_cd": "NAICS code as recorded (TM customers only)",
    "naics_desc": "NAICS description",
    "naics2": "2-digit NAICS sector, only when naics_status = valid",
    "naics_status": "valid / placeholder (******, UNKNOWN, …) / missing",
    "is_tm": "true when the customer has a customer_pwr_id",
    "family_size": "number of customers sharing this master_pwr_id",
    "customer_start_dt": "customer start date",
    "in_cust_dim": "false when the mdm_id is absent from neo4j_customer (identity columns then null)",
    "month": "partition: activity month YYYY-MM"}
SCOPE_TXT = {"int": "other PNC customers", "ext": "external counterparties", "self": "the customer's own accounts (same mdm_id)",
             "all": "PNC customers + external counterparties (self-transfers excluded)"}
DIR_TXT = {"in": "received", "out": "paid"}
METRIC_SPEC = []   # (column, family, scope, direction, definition)
for d in D:
    for s in ["int", "ext", "self", "all"]:
        METRIC_SPEC.append((f"amt_{s}_{d}", "flow", s, d, f"dollars {DIR_TXT[d]} with {SCOPE_TXT[s]}"))
        METRIC_SPEC.append((f"ntxn_{s}_{d}", "flow", s, d, f"transactions {DIR_TXT[d]} with {SCOPE_TXT[s]}"))
    for s in ["int", "ext", "all"]:
        METRIC_SPEC.append((f"ncpty_{s}_{d}", "breadth", s, d, f"distinct counterparties {DIR_TXT[d]} with, among {SCOPE_TXT[s]} (external at entity level ent_l1)"))
    METRIC_SPEC += [
        (f"active_days_{d}", "flow", "all", d, f"distinct posting dates with at least one transaction {DIR_TXT[d]}"),
        (f"ext_share_{d}", "flow", "all", d, f"share of dollars {DIR_TXT[d]} that involve external counterparties"),
        (f"top1_share_{d}", "concentration", "all", d, f"share of dollars {DIR_TXT[d]} with the single largest counterparty"),
        (f"hhi_{d}", "concentration", "all", d, f"Herfindahl index of dollars {DIR_TXT[d]} across counterparties (1 = one counterparty)"),
        (f"n_fi_{d}", "banks", "ext", d, f"distinct external banks of counterparties {DIR_TXT[d]} with"),
        (f"top_fi_share_{d}", "banks", "ext", d, f"share of bank-identified external dollars {DIR_TXT[d]} going through the largest bank"),
        (f"fi_coverage_{d}", "banks", "ext", d, f"share of external dollars {DIR_TXT[d]} whose counterparty bank is known"),
        (f"ext_business_share_{d}", "composition", "ext", d, f"share of external dollars {DIR_TXT[d]} with business-like counterparty names"),
        (f"int_tm_share_{d}", "composition", "int", d, f"share of internal dollars {DIR_TXT[d]} with TM customers"),
        (f"int_org_share_{d}", "composition", "int", d, f"share of internal dollars {DIR_TXT[d]} with organisations (party_type O)"),
        (f"n_naics2_{d}", "sector", "int", d, f"distinct NAICS2 sectors of internal counterparties {DIR_TXT[d]} with (null below {MIN_NAICS_CP} counterparties with valid NAICS)"),
        (f"naics2_top_share_{d}", "sector", "int", d, f"dollar share of the largest counterparty sector, internal, {DIR_TXT[d]} (same guard)"),
        (f"naics2_entropy_{d}", "sector", "int", d, f"Shannon entropy (nats) of dollars across counterparty sectors, internal, {DIR_TXT[d]} (same guard)"),
        (f"same_naics2_share_{d}", "sector", "int", d, f"share of sector-known internal dollars {DIR_TXT[d]} with counterparties in the customer's own NAICS2"),
    ]
    METRIC_SPEC += [(f"share_{r}_{d}", "rails", "all", d, f"share of dollars {DIR_TXT[d]} on {r}") for r in RAILS]
    METRIC_SPEC += [
        (f"n_rails_{d}", "rails", "all", d, f"number of rails used for dollars {DIR_TXT[d]}"),
        (f"new_cpty_{d}", "churn", "all", d, f"counterparties {DIR_TXT[d]} with this month but not last month (null when last month is unavailable)"),
        (f"retained_cpty_{d}", "churn", "all", d, f"counterparties {DIR_TXT[d]} with both this month and last month"),
        (f"lost_cpty_{d}", "churn", "all", d, f"counterparties {DIR_TXT[d]} with last month but not this month"),
        (f"new_amt_share_{d}", "churn", "all", d, f"share of dollars {DIR_TXT[d]} that went to/came from new counterparties"),
    ]
METRIC_SPEC += [
    ("net_flow_all", "flow", "all", "", "amt_all_in − amt_all_out"),
    ("net_flow_ext", "flow", "ext", "", "amt_ext_in − amt_ext_out"),
    ("passthrough_ratio", "flow", "all", "", "min(amt_all_in, amt_all_out) / max(...): 1 = everything received is paid out"),
]
DICT = pd.DataFrame([{"column": c, "family": "identity", "scope": "", "direction": "", "definition": IDENTITY_DEF[c]}
                     for c in IDENTITY_COLS + ["month"]] +
                    [{"column": c, "family": f, "scope": s, "direction": d, "definition": t} for c, f, s, d, t in METRIC_SPEC])
write_full(spark.createDataFrame(DICT), "cust_metrics_dictionary")
print(f"{len(IDENTITY_COLS)} identity + {len(METRIC_SPEC)} metric columns; dictionary written to {T['cust_metrics_dictionary']}")

In [ ]:
MONTH_LOG = []
EM = spark.read.parquet(P["edge_monthly"])
for m in RUN_MONTHS:
    if not FORCE and month_done("cust_metrics_monthly", m):
        print(f"{m}: already built, skipped"); continue
    t0 = time.time()
    pm = prev_month(m)
    if not month_done("cust_pair_month", pm) and exists(f"{P['edge_monthly']}/month={pm}"):
        with timed("pairs for previous month (churn warm-up)", pm):
            build_pairs(pm, perspective(pm))
    persp = perspective(m).persist(MD)
    with timed("perspective rows + pairs", m):
        if FORCE or not month_done("cust_pair_month", m):
            build_pairs(m, persp)
    PR = spark.read.parquet(f"{P['cust_pair_month']}/month={m}")
    with timed("metrics", m):
        write_month(metrics_for(m, persp, PR), "cust_metrics_monthly", m, "mdm_id", ["mdm_id"])
    persp.unpersist()

    with timed("reconciliation", m):
        mt = spark.read.parquet(f"{P['cust_metrics_monthly']}/month={m}")
        got = mt.agg(*[F.sum(F.col(f"amt_int_{d}") + F.col(f"amt_ext_{d}") + F.col(f"amt_self_{d}")).alias(d) for d in D],
                     F.count("*").alias("customers"), F.sum(F.col("is_tm").cast("int")).alias("tm_customers"),
                     F.sum((~F.col("in_cust_dim")).cast("int")).alias("not_in_cust_dim")).first()
        e = EM.filter(F.col("month") == m)
        exp = e.agg(F.sum(F.when(F.col("src_kind") == "CUST", F.col("amount"))).alias("out"),
                    F.sum(F.when(F.col("dst_kind") == "CUST", F.col("amount"))).alias("in")).first()
        ok = (got["out"] == exp["out"]) and (got["in"] == exp["in"])
    MONTH_LOG.append({"month": m, "customers": got["customers"], "tm_customers": got["tm_customers"],
                      "not_in_cust_dim": got["not_in_cust_dim"], "out_usd": float(got["out"] or 0),
                      "in_usd": float(got["in"] or 0), "ok": ok, "minutes": (time.time() - t0) / 60})
    with open("pkg_cust_metrics_runlog.csv", "a") as f:
        f.write(f"{dt.datetime.now().isoformat(timespec='seconds')},{m},{(time.time()-t0):.0f},{got['customers']},{ok}\n")
    print(f"{m}: done in {(time.time()-t0)/60:,.1f} min | reconciliation {'OK' if ok else 'MISMATCH ❌'}")
    if not ok:
        raise AssertionError(f"{m}: customer flows {got['out']}/{got['in']} ≠ edge totals {exp['out']}/{exp['in']}")

## §3 QA, storage, and a first look at the TM population

In [ ]:
if MONTH_LOG:
    show(pd.DataFrame(MONTH_LOG), "Per month: customers, reconciliation, time", money=("out_usd", "in_usd"))

built = [m for m in RUN_MONTHS if month_done("cust_metrics_monthly", m)]
st = []
for k in ("cust_pair_month", "cust_metrics_monthly"):
    tot = sum(size_of(f"{P[k]}/month={m}")[0] for m in built)
    st.append({"table": T[k], "months": len(built), "GB_total": tot / GB, "GB_per_month": tot / max(len(built), 1) / GB})
show(pd.DataFrame(st), "Storage")

if built:
    lm = built[-1]
    mt = spark.read.parquet(f"{P['cust_metrics_monthly']}/month={lm}").filter(F.col("is_tm"))
    look = ["ncpty_all_out", "ncpty_all_in", "ncpty_ext_out", "n_fi_out", "active_days_out", "top1_share_out", "hhi_out",
            "ext_share_out", "share_ACH_out", "share_CHECK_out", "share_WIRE_out", "share_RTP_PRT_out",
            "int_tm_share_out", "same_naics2_share_out", "passthrough_ratio", "new_cpty_out", "lost_cpty_out"]
    rows = []
    for col in look:
        r = mt.agg(F.expr(f"percentile_approx({col}, array(0.1, 0.5, 0.9))").alias("q"),
                   F.avg(F.col(col).isNull().cast("double")).alias("null_share")).first()
        q = r["q"] or [None, None, None]
        rows.append({"metric": col, "p10": q[0], "p50": q[1], "p90": q[2], "null_share": r["null_share"]})
    show(pd.DataFrame(rows), f"TM customers, {lm}: distribution sanity check", pct=("null_share",))
    show(spark.table(T["cust_metrics_dictionary"]).toPandas(), "Data dictionary")

In [ ]:
print("-- Impala: run once per new table, then REFRESH after every Spark run")
for k in ("cust_dim", "cust_pair_month", "cust_metrics_monthly", "cust_metrics_dictionary"):
    print(f"INVALIDATE METADATA {T[k]};")
for k in ("cust_dim", "cust_pair_month", "cust_metrics_monthly", "cust_metrics_dictionary"):
    print(f"REFRESH {T[k]};")
CD.unpersist()